# Public Transit Honor System vs. Fare Gates

Michael Tian, Michael Mehall, Evan Crow, Leandros Manwaring, Ian Solberg 

April 2026

---

In [59]:
import pandas as pd
import numpy as np
from Research_Framework.ResearchHandler import ResearchHandler
import Research_Framework.transforms as tf
from mbta_handling import blank, build_policy_flow 

---

#### Setup Cell, intializing source data.

In [60]:
FROM_DATA_SOURCES = False
if FROM_DATA_SOURCES:
    df = build_policy_flow(nodes_fp="data/mbta_rapid_transit/MBTA_NODE.shp", 
                        gse_fp="data/GSE/GSE.csv", 
                        travel_times_hr_fp="data/TravelTimes_2026/2026-02_HRTravelTimes.csv", 
                        travel_time_lr_fp="data/TravelTimes_2026/2026-02_LRTravelTimes.csv")
    mbta = ResearchHandler(df, handling_function=blank, initialize=False)
    mbta.data.to_csv("data/output_data/mbta_gate_flows_clean.csv", index=False)
    mbta.data.head()
else:
    fp = "data/output_data/mbta_gate_flows_clean.csv"
    mbta = ResearchHandler(fp, handling_function=blank, initialize=True)
    mbta.data.head()

---

# MBTA Policy Flow Dataset

**18,466 rows** — each row is a directed station pair on a rapid transit line during a time-of-day bucket, with average travel time and fare gate activity for February 2026.

## Columns

| Column | Description |
|--------|-------------|
| `station_line_source` | Origin station and line (e.g. `Kendall/MIT, RED`) |
| `station_line_destination` | Destination station and line |
| `line` | Rapid transit line: RED, ORANGE, or BLUE |
| `time_bucket` | Period of day: Early Morning, AM Rush, Midday, PM Rush, Evening, Late Night |
| `avg_travel_time_sec` | Mean travel time in seconds across all February 2026 trips |
| `source_avg_gated_entries` | Mean fare gate entries at origin station during that time bucket |
| `dest_avg_gated_entries` | Mean fare gate entries at destination station during that time bucket |

## Time Buckets

| Bucket | Hours |
|--------|-------|
| Early Morning | 5:00–7:00 |
| AM Rush | 7:00–10:00 |
| Midday | 10:00–15:00 |
| PM Rush | 15:00–19:00 |
| Evening | 19:00–22:00 |
| Late Night | 22:00–5:00 |

## Notes

Full Description of datasources and transformation can be found in `data/README.md`

---

# Variable Engineering 

In [ ]:
mbta.data["short_haul"] = (mbta.data["avg_travel_time_sec"] < 900).astype(int) # Short haul indicator (Barabino et al., 2015 — OR=0.7065, p=0.008)
mbta.data["source_station"] = mbta.data["station_line_source"].str.rsplit(", ", n=1).str[0] # Isolate source
mbta.data["dest_station"] = mbta.data["station_line_destination"].str.rsplit(", ", n=1).str[0] # Isolate destination

line_flags = ( # Line Dummies (Binary RED ORANGE BLUE GREEN)
    mbta.data
    .groupby("source_station")["line"]
    .apply(lambda x: x.unique())
    .explode()
    .str.get_dummies()
)
line_flags = line_flags.groupby(level=0).max()

station_panel = ( # to collapse by station
    mbta.data
    .groupby(["source_station", "time_bucket"])
    .agg(
        avg_gated_entries=("source_avg_gated_entries", "mean"),
        short_haul_exposure=("short_haul", "mean"),
        n_destinations=("dest_station", "nunique"),
    )
    .reset_index()
)
station_panel = station_panel.merge(line_flags, left_on="source_station", right_index=True, how="left") # line dummy merge
bucket_dummies = pd.get_dummies(station_panel["time_bucket"], prefix="bucket").astype(float) 
bucket_dummies = bucket_dummies.drop(columns=["bucket_Midday"]) # Midday is reference (Prevent Multicollinearity - not an issue for lines since 1 station can serve multiple lines)
station_panel = pd.concat([station_panel, bucket_dummies], axis=1) # time bucket dummy merge
station_panel.to_csv(path_or_buf="data/output_data/toregress.csv") # save in current state before regression

np.float64(328.47262092084617)

In [ ]:
import statsmodels.api as sm

analysis = ResearchHandler(source="data/output_data/toregress.csv", handling_function=blank)
# Naive
analysis.clear_caches()
analysis.set_dependent("avg_gated_entries")
analysis.add_independents("n_destinations")
analysis.add_controls(
    "BLUE", "GREEN", "ORANGE", "RED",
    "bucket_AM Rush", "bucket_Early Morning",
    "bucket_Evening", "bucket_Late Night", "bucket_PM Rush"
)
X_naive = sm.add_constant(analysis.get_X())
y = analysis.get_y()
naive_model = sm.OLS(y, X_naive).fit(cov_type="HC1")

# Informed — just add short_haul_exposure
analysis.add_independents("short_haul_exposure") # add in evasion proxy
X_informed = sm.add_constant(analysis.get_X())
informed_model = sm.OLS(y, X_informed).fit(cov_type="HC1")

Caches cleared
Dependent variable set to: avg_gated_entries
Independent variables: ['n_destinations']
Control variables: ['BLUE', 'GREEN', 'ORANGE', 'RED', 'bucket_AM Rush', 'bucket_Early Morning', 'bucket_Evening', 'bucket_Late Night', 'bucket_PM Rush']
Independent variables: ['n_destinations', 'short_haul_exposure']


## Model Summaries

In [45]:
print(naive_model.summary())
print(informed_model.summary())

                            OLS Regression Results                            
Dep. Variable:      avg_gated_entries   R-squared:                       0.434
Model:                            OLS   Adj. R-squared:                  0.425
Method:                 Least Squares   F-statistic:                     37.71
Date:                Mon, 30 Mar 2026   Prob (F-statistic):           1.93e-58
Time:                        13:48:56   Log-Likelihood:                -4958.2
No. Observations:                 666   AIC:                             9938.
Df Residuals:                     655   BIC:                             9988.
Df Model:                          10                                         
Covariance Type:                  HC1                                         
                           coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------------
const                  819.1557 

In [ ]:
beta_short_haul = informed_model.params["short_haul_exposure"]

# ── Naive allocation: demand share per bucket ──
naive = station_panel.pivot(index="source_station", columns="time_bucket", values="avg_gated_entries") # avg gated entries per bucket per station
naive = naive[["Early Morning", "AM Rush", "Midday", "PM Rush", "Evening", "Late Night"]]
naive = naive.div(naive.sum(axis=0), axis=1)


# ── Informed allocation: demand adjusted by short_haul_exposure
bucket_means = station_panel.groupby("time_bucket")["avg_gated_entries"].mean()
adj_weight = beta_short_haul / bucket_means

she = station_panel.pivot(index="source_station", columns="time_bucket", values="short_haul_exposure")
she = she[["Early Morning", "AM Rush", "Midday", "PM Rush", "Evening", "Late Night"]]

informed = naive * (1 + adj_weight * she) # adjust naive with informed weighting 
informed = informed.div(informed.sum(axis=0), axis=1)


# ── Delta ──
delta = informed - naive


naive["avg"] = naive.mean(axis=1)
naive = naive.sort_values("avg", ascending=False)
naive.to_csv("data/output_data/naive_allocation.csv")

informed["avg"] = informed.mean(axis=1)
informed = informed.sort_values("avg", ascending=False)
informed.to_csv("data/output_data/informed_allocation.csv")

delta["avg"] = delta.mean(axis=1)
delta = delta.sort_values("avg", ascending=False)
delta.to_csv("data/output_data/allocation_delta.csv")